# 02 · Train the emotion head (C4)

**Goal:** learn to predict **valence** (negative ↔ positive) and **arousal** (calm ↔ energetic) for each 10 s window from its CLAP embedding, and show that my model beats simpler ones.

Models compared (this is the **ablation**: remove one part at a time and see what it was worth):

| Model | Looks at | Idea |
|---|---|---|
| Mean predictor | nothing | always predicts the training average (the "do nothing" floor) |
| **Baseline:** ridge regression | one window | a straight-line (linear) mapping from 512 numbers → 2 numbers |
| BiLSTM | the whole song, in order | reads windows forwards **and** backwards, so each prediction knows what comes before and after |
| **BiLSTM + attention** (my model) | the whole song | adds attention: each window can look directly at the most relevant other windows |

Metrics, computed on a **test set of songs the models never saw**:
- **RMSE / MAE**: average size of the error (lower = better)
- **Pearson r**: do predictions go up and down together with the truth? (higher = better)
- **CCC** (Concordance Correlation Coefficient): like Pearson, but also punishes being shifted or squashed; the standard metric in emotion recognition (higher = better, 1 = perfect)

**Input:** `My Drive/C4/embeddings/deam_clap_windows.npz` from notebook 01. **Runtime:** T4 GPU.

## 1. Connect Drive and load the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/C4'
DATA_PATH = f'{DRIVE_DIR}/embeddings/deam_clap_windows.npz'
OUT_DIR = f'{DRIVE_DIR}/training'

In [ ]:
import os, json, random
import numpy as np
import torch

os.makedirs(OUT_DIR, exist_ok=True)
d = np.load(DATA_PATH)
X, V, A = d['embeddings'], d['valence'], d['arousal']
SONG, START = d['song_id'], d['start_s']
Y = np.stack([V, A], axis=1)                 # column 0 = valence, column 1 = arousal
print('windows:', X.shape, '| songs:', len(np.unique(SONG)))

## 2. Fix the random seeds (reproducibility)
Training uses randomness (initial weights, shuffling). Fixing the seeds means running the notebook again gives the same numbers, so my results can be checked.

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 3. Split by song: 70 % train / 15 % validation / 15 % test
Windows overlap (5 s hop) and windows of the same song sound alike. If one song's windows were in both train and test, the model would be tested on audio it has practically already seen: **data leakage**, giving falsely good scores. So whole songs go to exactly one set.

- **train**: the model learns from these
- **validation**: used to choose settings and to stop training at the right moment
- **test**: touched only once, at the very end, for the final honest score

In [ ]:
songs = np.unique(SONG)
rng = np.random.default_rng(SEED)
rng.shuffle(songs)
n_train, n_val = int(0.70 * len(songs)), int(0.15 * len(songs))
split_songs = {'train': songs[:n_train], 'val': songs[n_train:n_train + n_val], 'test': songs[n_train + n_val:]}
mask = {k: np.isin(SONG, v) for k, v in split_songs.items()}

assert not set(split_songs['train']) & set(split_songs['test']), 'a song is in train AND test!'
for k in mask:
    print(f'{k:5s}: {len(split_songs[k]):4d} songs, {mask[k].sum():5d} windows')

## 4. Metrics

In [ ]:
def ccc(y_true, y_pred):
    # Concordance Correlation Coefficient: 1 = perfect, 0 = no agreement
    mt, mp = y_true.mean(), y_pred.mean()
    vt, vp = y_true.var(), y_pred.var()
    cov = ((y_true - mt) * (y_pred - mp)).mean()
    return 2 * cov / (vt + vp + (mt - mp) ** 2)

def evaluate(y_true, y_pred):
    out = {}
    for i, name in enumerate(['valence', 'arousal']):
        t, p = y_true[:, i], y_pred[:, i]
        out[f'{name}_RMSE'] = float(np.sqrt(np.mean((t - p) ** 2)))
        out[f'{name}_MAE'] = float(np.mean(np.abs(t - p)))
        out[f'{name}_Pearson'] = float(np.corrcoef(t, p)[0, 1]) if p.std() > 1e-8 else 0.0   # a constant prediction has no correlation
        out[f'{name}_CCC'] = float(ccc(t, p))
    return out

results = {}   # model name -> test metrics

## 5. Reference: mean predictor
Always predicts the average training label. Any useful model must beat this; Pearson/CCC are 0 by definition.

In [ ]:
mean_pred = np.tile(Y[mask['train']].mean(axis=0), (mask['test'].sum(), 1))
results['Mean predictor'] = evaluate(Y[mask['test']], mean_pred)
results['Mean predictor']

## 6. Baseline: ridge regression (per window, no sequence context)
A linear model: `prediction = embedding · W + b`. "Ridge" adds a penalty `alpha` on large weights so it doesn't overfit 512 inputs. Solved directly with one formula, no training loop:

`W = (XᵀX + alpha·I)⁻¹ Xᵀy`

We try several `alpha` values and keep the one with the best **validation** CCC.

In [ ]:
def fit_ridge(X, y, alpha):
    Xb = np.hstack([X, np.ones((len(X), 1))])          # extra column of 1s = the bias b
    reg = alpha * np.eye(Xb.shape[1]); reg[-1, -1] = 0  # don't penalise the bias
    return np.linalg.solve(Xb.T @ Xb + reg, Xb.T @ y)

def predict_ridge(W, X):
    return np.hstack([X, np.ones((len(X), 1))]) @ W

best = None
for alpha in [0.01, 0.1, 1, 10, 100]:
    W = fit_ridge(X[mask['train']], Y[mask['train']], alpha)
    m = evaluate(Y[mask['val']], predict_ridge(W, X[mask['val']]))
    score = (m['valence_CCC'] + m['arousal_CCC']) / 2
    print(f'alpha={alpha:<6} val mean CCC = {score:.3f}')
    if best is None or score > best[0]:
        best = (score, alpha, W)

_, RIDGE_ALPHA, W_ridge = best
results['Ridge (baseline)'] = evaluate(Y[mask['test']], np.clip(predict_ridge(W_ridge, X[mask['test']]), -1, 1))
np.savez(f'{OUT_DIR}/c4_ridge_baseline.npz', W=W_ridge, alpha=RIDGE_ALPHA)
print('best alpha:', RIDGE_ALPHA)
results['Ridge (baseline)']

## 7. Sequences: one song = one ordered list of windows
The BiLSTM reads a whole song at once. Songs have different numbers of windows, so in a batch shorter songs are **padded** with zeros, and a **mask** remembers which positions are real, so padding never counts in the loss or in attention.

In [ ]:
def build_sequences(split):
    seqs = []
    for s in split_songs[split]:
        idx = np.where(SONG == s)[0]
        idx = idx[np.argsort(START[idx])]               # windows in time order
        seqs.append((torch.tensor(X[idx]), torch.tensor(Y[idx])))
    return seqs

def make_batch(seqs):
    T = max(len(x) for x, _ in seqs)
    xb = torch.zeros(len(seqs), T, X.shape[1]); yb = torch.zeros(len(seqs), T, 2)
    mb = torch.zeros(len(seqs), T, dtype=torch.bool)
    for i, (x, y) in enumerate(seqs):
        xb[i, :len(x)], yb[i, :len(y)], mb[i, :len(x)] = x, y, True
    return xb.to(device), yb.to(device), mb.to(device)

seq = {k: build_sequences(k) for k in ['train', 'val', 'test']}
lengths = [len(x) for x, _ in seq['train']]
print('train sequences:', len(lengths), '| windows per song: min', min(lengths), 'max', max(lengths))

## 8. The model: BiLSTM (+ optional attention)
- **LSTM** (Long Short-Term Memory): a neural network that reads a sequence step by step and keeps a "memory", so the prediction for window 5 can use what it heard in windows 1–4.
- **Bi**directional: one LSTM reads forwards, one backwards; their outputs are joined, so each window also knows what comes **after** it.
- **Attention**: for every window, the model computes a weight for every other window ("how relevant are you to me?") and mixes in a weighted summary. It can then link, e.g., a quiet intro to a loud chorus directly, without passing through every step in between.
- **tanh** at the end squeezes outputs into [-1, 1], the same range as the labels.
- **Dropout** randomly switches off some inputs during training, which reduces overfitting (memorising the training songs).

`use_attention=False` gives the plain BiLSTM for the ablation; everything else is identical.

In [ ]:
import torch.nn as nn

class EmotionHead(nn.Module):
    def __init__(self, input_dim=512, hidden=128, use_attention=True, dropout=0.3):
        super().__init__()
        self.use_attention = use_attention
        self.dropout = nn.Dropout(dropout)
        self.bilstm = nn.LSTM(input_dim, hidden, batch_first=True, bidirectional=True)
        if use_attention:
            self.attention = nn.MultiheadAttention(2 * hidden, num_heads=1, batch_first=True)
            self.norm = nn.LayerNorm(2 * hidden)
        self.out = nn.Linear(2 * hidden, 2)          # 2 outputs: valence, arousal

    def forward(self, x, mask):
        # x: (batch, windows, 512), mask: (batch, windows) True = real window
        h, _ = self.bilstm(self.dropout(x))          # (batch, windows, 2*hidden)
        if self.use_attention:
            context, weights = self.attention(h, h, h, key_padding_mask=~mask)
            h = self.norm(h + context)               # keep the BiLSTM info and add the attention summary
        return torch.tanh(self.out(self.dropout(h)))

print(EmotionHead())
print('trainable parameters:', sum(p.numel() for p in EmotionHead().parameters()))

## 9. Training loop
- **Loss**: mean squared error over real (non-padded) windows only.
- **Adam** optimiser adjusts the weights a little after every batch.
- **Early stopping**: after each epoch (one pass over all training songs) we check the validation loss. We keep the best version and stop when it hasn't improved for `PATIENCE` epochs. That's the point where the model starts memorising instead of learning.

In [ ]:
HPARAMS = dict(hidden=128, dropout=0.3, lr=1e-3, weight_decay=1e-4, batch_size=32, max_epochs=100, patience=12)

def masked_mse(pred, y, m):
    return ((pred - y) ** 2)[m].mean()

def predict_seqs(model, seqs):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(seqs), 64):
            xb, _, mb = make_batch(seqs[i:i + 64])
            p = model(xb, mb)
            preds += [p[j, :len(seqs[i + j][0])].cpu().numpy() for j in range(len(p))]
    return np.concatenate(preds), np.concatenate([y.numpy() for _, y in seqs])

def train_model(use_attention, name):
    torch.manual_seed(SEED)
    model = EmotionHead(hidden=HPARAMS['hidden'], use_attention=use_attention, dropout=HPARAMS['dropout']).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=HPARAMS['lr'], weight_decay=HPARAMS['weight_decay'])
    history, best_val, best_state, waited = {'train': [], 'val': []}, float('inf'), None, 0
    rng = random.Random(SEED)
    for epoch in range(HPARAMS['max_epochs']):
        model.train()
        order = list(range(len(seq['train']))); rng.shuffle(order)
        losses = []
        for i in range(0, len(order), HPARAMS['batch_size']):
            xb, yb, mb = make_batch([seq['train'][j] for j in order[i:i + HPARAMS['batch_size']]])
            loss = masked_mse(model(xb, mb), yb, mb)
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        p, t = predict_seqs(model, seq['val'])
        val_loss = float(np.mean((p - t) ** 2))
        history['train'].append(float(np.mean(losses))); history['val'].append(val_loss)
        if val_loss < best_val:
            best_val, waited = val_loss, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
        if epoch % 5 == 0:
            print(f'{name} epoch {epoch:3d}  train {history["train"][-1]:.4f}  val {val_loss:.4f}')
        if waited >= HPARAMS['patience']:
            print(f'{name}: early stop at epoch {epoch}, best val loss {best_val:.4f}')
            break
    model.load_state_dict(best_state)
    return model, history

## 10. Train BiLSTM (no attention), then BiLSTM + attention

In [ ]:
bilstm, hist_bilstm = train_model(use_attention=False, name='BiLSTM')
p, t = predict_seqs(bilstm, seq['test'])
results['BiLSTM'] = evaluate(t, p)
results['BiLSTM']

In [ ]:
bilstm_attn, hist_attn = train_model(use_attention=True, name='BiLSTM+attention')
p, t = predict_seqs(bilstm_attn, seq['test'])
results['BiLSTM + attention'] = evaluate(t, p)
results['BiLSTM + attention']

## 11. Loss curves (for the slides)
Train loss should go down. Validation loss going down then flattening/rising shows where early stopping kicked in.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (name, h) in zip(axes, [('BiLSTM', hist_bilstm), ('BiLSTM + attention', hist_attn)]):
    ax.plot(h['train'], label='train'); ax.plot(h['val'], label='validation')
    ax.set(title=name, xlabel='epoch', ylabel='MSE loss'); ax.legend()
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/loss_curves.png', dpi=150); plt.show()

## 12. Comparison table (the ablation result)
All rows are on the same **test songs**. ↓ = lower is better, ↑ = higher is better.

In [ ]:
import pandas as pd

table = pd.DataFrame(results).T
cols = [f'{d}_{m}' for d in ['valence', 'arousal'] for m in ['RMSE', 'MAE', 'Pearson', 'CCC']]
table = table[cols].round(3)
table.to_csv(f'{OUT_DIR}/model_comparison.csv')

fig, ax = plt.subplots(figsize=(14, 0.6 + 0.45 * len(table))); ax.axis('off')
labels = [c.replace('_', ' ') + (' ↓' if c.endswith(('RMSE', 'MAE')) else ' ↑') for c in cols]
ax.table(cellText=table.values, rowLabels=table.index, colLabels=labels, loc='center').scale(1, 1.4)
plt.savefig(f'{OUT_DIR}/model_comparison.png', dpi=150, bbox_inches='tight'); plt.show()
table

## 13. Save the model for the backend
The backend (Day 10, `RealEmotionModel`) loads `c4_emotion_head.pt`. Besides the weights, the file stores the settings needed to rebuild the exact same model, the hyperparameters and the test scores.

**Plan (CLAUDE.md):** ship BiLSTM + attention. **Fallback:** if it does *not* beat the ridge baseline on mean test CCC, I ship the baseline instead and say so honestly. The cell below prints which one wins.

In [ ]:
def mean_ccc(name):
    return (results[name]['valence_CCC'] + results[name]['arousal_CCC']) / 2

for name in results:
    print(f'{name:22s} mean test CCC = {mean_ccc(name):.3f}')

torch.save({
    'state_dict': bilstm_attn.state_dict(),
    'config': dict(input_dim=512, hidden=HPARAMS['hidden'], use_attention=True, dropout=HPARAMS['dropout']),
    'head_name': 'bilstm_attention_v1',
    'hparams': HPARAMS, 'seed': SEED, 'test_metrics': results['BiLSTM + attention'],
}, f'{OUT_DIR}/c4_emotion_head.pt')

with open(f'{OUT_DIR}/training_log.json', 'w') as f:
    json.dump(dict(hparams=HPARAMS, seed=SEED, ridge_alpha=RIDGE_ALPHA,
                   split_sizes={k: int(len(v)) for k, v in split_songs.items()},
                   epochs=dict(bilstm=len(hist_bilstm['val']), bilstm_attention=len(hist_attn['val'])),
                   test_results=results), f, indent=2)

winner = 'BiLSTM + attention' if mean_ccc('BiLSTM + attention') >= mean_ccc('Ridge (baseline)') else 'Ridge (baseline)'
print('\nSaved c4_emotion_head.pt, c4_ridge_baseline.npz, training_log.json, figures and CSV to', OUT_DIR)
print('Best on test (mean CCC):', winner)